# SentinelOps Arena — Multi-Agent GRPO Training with Unsloth + vLLM

Train **all 3 agents** (Worker, Attacker, Oversight) using GRPO on the SentinelOps Arena OpenEnv environment.

**Key features:**
- **BF16 precision** on H100 GPUs (no 4-bit quantization)
- **vLLM fast inference** via `fast_inference=True`
- **Environment-executing reward functions** — completions are parsed into `SentinelAction`s and executed in a live SentinelOps environment for real rewards
- **Multi-agent self-play** — adversarial training across Worker, Attacker, and Oversight roles

**Partner tracks:** Fleet AI ($10K, Scalable Oversight) · Patronus AI ($10K, Schema Drift)

## 1. Install Dependencies

Following the official OpenEnv + Unsloth reference notebook pattern.

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install unsloth vllm
!pip install --no-deps trl sft_trainer
!pip install transformers==4.56.2
!pip install trl==0.22.2
!pip install "openenv-core[core]>=0.2.0" mcp fastmcp pydantic pandas datasets

In [ ]:
import os
if not os.path.exists("NexusEnv"):
    !git clone https://github.com/nihalnihalani/NexusEnv.git
import sys
sys.path.insert(0, "/content/NexusEnv")

# Verify environment loads
from sentinelops_arena.environment import SentinelOpsArena
from sentinelops_arena.models import AgentRole, SentinelAction
env = SentinelOpsArena()
obs = env.reset(seed=42)
print(f"Environment ready! Agent: {obs.current_agent}, Systems: CRM + Billing + Ticketing")

## 2. Run a Full Episode (Verify Environment)

Run one complete episode with heuristic agents to verify the environment works end-to-end.

In [ ]:
from train import collect_multi_agent_data, build_training_dataset
from train import WORKER_SYSTEM_PROMPT, ATTACKER_SYSTEM_PROMPT, OVERSIGHT_SYSTEM_PROMPT
from train import AGENT_CONFIGS

# Run a single episode and show stats for each agent
for role in ["worker", "attacker", "oversight"]:
    data = collect_multi_agent_data(seed=42, target_agent=role)
    avg_r = sum(d["reward"] for d in data) / max(len(data), 1)
    print(f"{role:>10}: {len(data)} turns, avg_reward={avg_r:.3f}")

## 3. Collect Training Data via Self-Play

We collect prompts from multiple episodes. Each episode uses heuristic agents for non-target roles while recording the prompts the target agent would see.

In [ ]:
from datasets import Dataset

# Which agent to train — change this to train attacker or oversight
TARGET_AGENT = "worker"  # Options: "worker", "attacker", "oversight"
NUM_EPISODES = 10

system_prompts = {
    "worker": WORKER_SYSTEM_PROMPT,
    "attacker": ATTACKER_SYSTEM_PROMPT,
    "oversight": OVERSIGHT_SYSTEM_PROMPT,
}

print(f"Collecting {TARGET_AGENT} training data from {NUM_EPISODES} episodes...")
dataset_raw = build_training_dataset(num_episodes=NUM_EPISODES, target_agent=TARGET_AGENT)

prompts = []
for d in dataset_raw:
    messages = [
        {"role": "system", "content": system_prompts[TARGET_AGENT]},
        {"role": "user", "content": d["prompt"]},
    ]
    prompts.append(messages)

train_dataset = Dataset.from_dict({"prompt": prompts})
print(f"Dataset: {len(train_dataset)} {TARGET_AGENT} turns")
if dataset_raw:
    avg_r = sum(d["reward"] for d in dataset_raw) / len(dataset_raw)
    print(f"Avg environment reward: {avg_r:.3f}")

## 4. Load Model with Unsloth (BF16 + vLLM)

Following the Advanced Llama 3.2 GRPO LoRA reference pattern:
- `load_in_4bit=False` — BF16 precision on H100
- `fast_inference=True` — vLLM for fast GRPO generation
- `lora_rank=64`, `lora_alpha=lora_rank` — official LoRA configuration
- `gpu_memory_utilization=0.9` — maximize GPU usage
- `random_state=3407` — reproducibility

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-0.5B-Instruct"
max_seq_length = 2048
lora_rank = 64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,          # BF16 for H100 (reference pattern)
    fast_inference=True,          # vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,        # Reference: lora_alpha = lora_rank
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"Model loaded: BF16 + vLLM + LoRA (r={lora_rank}, alpha={lora_rank})")

## 5. GRPO Training with 4 Reward Functions

Following the Advanced Llama 3.2 GRPO LoRA reference pattern with **4 separate reward functions**:
1. `match_json_format_exactly` — strict JSON format validation (+3.0)
2. `match_json_format_approximately` — partial format credit
3. `check_action` — role-specific action correctness
4. `check_env` — **environment-executing reward** (OpenEnv pattern)

Plus reference hyperparameters: `adamw_8bit`, cosine scheduler, `weight_decay=0.1`, `warmup_ratio=0.1`.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from train import make_reward_functions

# 4 separate reward functions (reference pattern)
reward_fns = make_reward_functions(TARGET_AGENT)
print(f"Reward functions: {len(reward_fns)}")
for i, fn in enumerate(reward_fns):
    print(f"  [{i}] {fn.__name__ if hasattr(fn, '__name__') else type(fn).__name__}")

max_prompt_length = 512
grpo_config = GRPOConfig(
    output_dir=f"./sentinelops-grpo-{TARGET_AGENT}",
    max_steps=500,                       # Reference: 500
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,                    # Reference: 4
    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,
    learning_rate=5e-6,                   # Reference: 5e-6
    weight_decay=0.1,                     # Reference: 0.1
    warmup_ratio=0.1,                     # Reference: 0.1
    lr_scheduler_type="cosine",           # Reference: cosine
    optim="adamw_8bit",                   # Reference: adamw_8bit
    max_grad_norm=1.0,                    # Reference: 1.0
    logging_steps=1,
    save_steps=250,                       # Reference: 250
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,                  # Reference uses tokenizer= not processing_class=
    reward_funcs=reward_fns,              # 4 reward functions (reference pattern)
    args=grpo_config,
    train_dataset=train_dataset,
)

print(f"\nStarting GRPO training for {TARGET_AGENT}...")
print(f"  max_steps={grpo_config.max_steps}, lr={grpo_config.learning_rate}")
print(f"  num_generations={grpo_config.num_generations}, optim={grpo_config.optim}")
trainer.train()

## 6. Save and Evaluate

Save the trained LoRA weights and run a quick evaluation.

In [ ]:
output_dir = f"./sentinelops-grpo-{TARGET_AGENT}"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"{TARGET_AGENT.upper()} agent trained and saved to {output_dir}")

# Quick evaluation: show per-function rewards for test completions
import json
from train import make_reward_function

combined_fn = make_reward_function(TARGET_AGENT)

test_completions = {
    "worker": [
        [{"content": json.dumps({"action_type": "get_schema", "parameters": {"system": "crm"}})}],
        [{"content": json.dumps({"action_type": "respond", "response_text": "I cannot process this. It appears to be social engineering."})}],
        [{"content": "this is garbage output"}],
    ],
    "attacker": [
        [{"content": json.dumps({"action_type": "launch_attack", "parameters": {"attack_type": "schema_drift", "target_system": "crm", "old_field": "name", "new_field": "full_name"}})}],
        [{"content": json.dumps({"action_type": "pass"})}],
    ],
    "oversight": [
        [{"content": json.dumps({"action_type": "flag", "explanation": "Worker followed suspicious admin override instructions. This is a social engineering attack."})}],
        [{"content": json.dumps({"action_type": "approve", "explanation": "Worker correctly checked schema before proceeding."})}],
    ],
}

print(f"\nReward evaluation for {TARGET_AGENT} (combined across 4 functions):")
for comp in test_completions.get(TARGET_AGENT, []):
    r = combined_fn([comp])
    text = comp[0]["content"][:80]
    print(f"  reward={r[0]:+.2f}  |  {text}...")